# Exp 07: v3 ablation

Card: [exp07_v3_ablation.md](../../experiments/exp07_v3_ablation.md)


In [ ]:
import os

import albumentations as A
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from PIL import Image

from core.augment import RECIPES, read_boxes
from core.config import IMG_SIZE, ROOT, SEED, TRAIN_DIR
from core.cv import VIP, CVResult, Config, CrossValidator
from core.data import read_train
from core.tracking import Experiment

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    OVERRIDES = {
        "experiment": "cnn-model-v3-ablation-smoke",
        "epochs": 1,
        "img_size": 64,
        "n_other": 20,
        "num_workers": 0,
        "retrain": True,
        "log_every_n_steps": 1,
    }
else:
    OVERRIDES = {}
CONFIG_DIR = ROOT / "sections" / "model" / "configs"
CONFIGS = {
    "box crop": Config.from_toml(
        CONFIG_DIR / "exp07_v3_ablation" / "resnet18_box_crop.toml", **OVERRIDES
    ),
    "new augmentations": Config.from_toml(
        CONFIG_DIR / "exp07_v3_ablation" / "resnet18_new_aug.toml", **OVERRIDES
    ),
}
# 2 x 2 corners trained in earlier experiments, only read
BASELINES = {
    "exp04": Config.from_toml(CONFIG_DIR / "exp04_oversampling_vip" / "resnet18.toml"),
    "v3 (exp06)": Config.from_toml(
        CONFIG_DIR / "exp06_augmentations_v3" / "resnet18.toml"
    ),
}
FIGURES = ROOT / "sections" / "model" / "figures"
PREFIX = "exp07_v3_ablation"
SAVE_FIGURES = not SMOKE

# dataviz reference palette, light mode; color follows entity across figures
SERIES = {
    "exp04": "#eb6834",
    "v3 (exp06)": "#2a78d6",
    "box crop": "#1baf7a",
    "new augmentations": "#eda100",
}
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#52514e",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.titlesize": "medium",
        "axes.titlecolor": "#52514e",
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "legend.frameon": False,
        "legend.fontsize": "small",
    }
)


## Draws


In [ ]:
N_DRAWS = 6
train_df = read_train()
image_ids = {
    cls: train_df[train_df[cls] == 1].sample(1, random_state=SEED)["image_id"].item()
    for cls in VIP
}
images = {
    cls: np.array(
        Image.open(TRAIN_DIR / f"{image_id}.jpg")
        .convert("RGB")
        .resize((IMG_SIZE, IMG_SIZE))
    )
    for cls, image_id in image_ids.items()
}
boxes = read_boxes()  # Grad-CAM pseudo-boxes, needed by BBoxSafeRandomCrop


def preview(tfms):
    """`tfms` without Normalize, ToTensorV2, keeping box handling if any."""
    bbox = tfms.processors.get("bboxes")
    return A.Compose(tfms.transforms[:-2], bbox_params=bbox.params if bbox else None)


def plot_boxes(ax, image, bboxes, labels, color):
    h, w = image.shape[:2]
    for (x0, y0, x1, y1), cls in zip(bboxes, labels):
        ax.add_patch(
            plt.Rectangle(
                (x0 * w, y0 * h),
                (x1 - x0) * w,
                (y1 - y0) * h,
                fill=False,
                ec=color,
                lw=1.5,
            )
        )
        ax.text(
            x0 * w + 2,
            y0 * h + 2,
            cls,
            color="white",
            fontsize=7,
            va="top",
            bbox=dict(fc=color, ec="none", pad=1),
        )


rows = [(cls, recipe) for cls in VIP for recipe in CONFIGS]
fig, axes = plt.subplots(
    len(rows), N_DRAWS + 1, figsize=(2.1 * (N_DRAWS + 1), 2.4 * len(rows))
)
for (cls, recipe), row_axes in zip(rows, axes):
    show = preview(RECIPES[CONFIGS[recipe].augment](IMG_SIZE).compose())
    with_boxes = "bboxes" in show.processors
    image_boxes = boxes.get(image_ids[cls], [])
    for i, ax in enumerate(row_axes):
        if i == 0:
            shown, bboxes, labels = (
                images[cls],
                [b for _, b in image_boxes],
                [c for c, _ in image_boxes],
            )
        elif with_boxes:
            out = show(
                image=images[cls],
                bboxes=np.array([b for _, b in image_boxes]).reshape(-1, 4),
                box_labels=[c for c, _ in image_boxes],
            )
            shown, bboxes, labels = out["image"], out["bboxes"], out["box_labels"]
        else:
            shown, bboxes, labels = show(image=images[cls])["image"], [], []
        ax.imshow(shown)
        if with_boxes:
            plot_boxes(ax, shown, bboxes, labels, SERIES[recipe])
        ax.set_axis_off()
        if (cls, recipe) == rows[0]:
            ax.set_title("Original" if i == 0 else f"Draw {i}", fontsize=9)
    row_axes[0].text(
        -0.08,
        0.5,
        f"{cls}\n{recipe}",
        transform=row_axes[0].transAxes,
        rotation=90,
        va="center",
        ha="right",
    )
fig.tight_layout()
figures = {f"{PREFIX}_draws.png": fig}
plt.show()

## Training


In [ ]:
results = {name: CVResult.load(cfg) for name, cfg in BASELINES.items()}

In [ ]:
results["box crop"] = CrossValidator(CONFIGS["box crop"]).run()

In [ ]:
results["new augmentations"] = CrossValidator(CONFIGS["new augmentations"]).run()

## Validation mAP per epoch


In [ ]:
MODELS = ["exp04", "box crop", "new augmentations", "v3 (exp06)"]
fig, ax = plt.subplots(figsize=(7, 4))
for name in MODELS:
    r = results[name]
    ax.plot(
        np.arange(1, r.config.epochs + 1),
        r.mean_curve("val_map"),
        color=SERIES[name],
        label=name,
    )
ax.set(title="Validation mAP, mean over folds 0 and 1", xlabel="epoch")
ax.legend(loc="lower right", title="model", alignment="left")
fig.tight_layout()
figures[f"{PREFIX}_val_map.png"] = fig
plt.show()

## AP comparison


In [ ]:
fold_ap_df = pd.concat(
    {name: results[name].fold_ap for name in MODELS}, names=["model"]
)
ap = fold_ap_df.groupby("model", sort=False).mean()
ap.round(3)

In [ ]:
# effect of each change against exp04; counts only if same sign on both folds
base = results["exp04"].fold_ap
effects = {}
for name in ["box crop", "new augmentations", "v3 (exp06)"]:
    diff = (results[name].fold_ap - base).T.add_prefix("fold ")
    diff["same sign"] = (diff > 0).all(axis=1) | (diff < 0).all(axis=1)
    effects[f"{name} − exp04"] = diff
effects = pd.concat(effects, axis=1)
# 0 when the two changes add up
effects[("interaction", "mean")] = (
    ap.loc["v3 (exp06)"]
    - ap.loc["box crop"]
    - ap.loc["new augmentations"]
    + ap.loc["exp04"]
)
effects.round(3)

In [ ]:
with Experiment(CONFIGS["box crop"].experiment).run("summary", {"val_mode": "fast"}) as run:
    mlflow.log_table(fold_ap_df.reset_index(), "fold_ap.json")
    flat = effects.copy()
    flat.columns = [" ".join(c) for c in flat.columns]
    mlflow.log_table(flat.reset_index(names="metric"), "effects.json")
    if SAVE_FIGURES:
        run.log_figures(figures, FIGURES)